# Data Preparation
Run this notebook **once** to build `data/syb_database.db`.

**Steps:**
1. Load all 53 Excel files into SQLite
2. Verify tables loaded correctly
3. Quick test query
4. Build metadata table

> Once done, you only need `data/syb_database.db` to run the agent.

In [7]:
# Step 1 — Load all Excel files into SQLite
import pandas as pd
import sqlite3
import glob
import os

DB_PATH = "data/syb_database.db"

def load_all_tables():
    excel_files = sorted(glob.glob("data/**/*.xlsx", recursive=True))
    conn = sqlite3.connect(DB_PATH)
    loaded, failed = [], []

    for path in excel_files:
        path = path.replace("\\", "/")
        if os.path.basename(path).startswith("~$"):  # skip Excel temp/lock files
            continue
        tname = os.path.splitext(os.path.basename(path))[0]
        try:
            df = pd.read_excel(path, sheet_name="Relational DB", skiprows=2)
            df.columns = df.columns.str.strip()
            for col in df.select_dtypes(include=["object"]).columns:
                df[col] = df[col].str.strip()
            df.replace(["Not Applicable", "لا ينطبق", "Not App", "n.a.", "N/A"], None, inplace=True)
            df.to_sql(tname, conn, if_exists="replace", index=False)
            loaded.append(tname)
            print(f"✅ {tname} — {len(df)} rows")
        except Exception as e:
            failed.append((path, str(e)))
            print(f"❌ {path}: {e}")

    conn.close()
    print(f"\nDone: {len(loaded)} loaded, {len(failed)} failed → {DB_PATH}")
    return loaded, failed

loaded, failed = load_all_tables()


✅ SYB_11_1_10 — 432 rows
✅ SYB_11_1_2 — 324 rows
✅ SYB_11_1_3 — 40 rows
✅ SYB_11_1_4 — 1296 rows
✅ SYB_11_3_1 — 54 rows
✅ SYB_11_3_2 — 150 rows
✅ SYB_11_3_3 — 90 rows
✅ SYB_13_1_1 — 432 rows
✅ SYB_13_1_2 — 540 rows
✅ SYB_13_1_3 — 432 rows
✅ SYB_13_1_4 — 288 rows
✅ SYB_13_1_5 — 168 rows
✅ SYB_13_1_6 — 2268 rows
✅ SYB_13_1_7 — 1512 rows
✅ SYB_14_6 — 144 rows
✅ SYB_14_7 — 117 rows
✅ SYB_14_8 — 44 rows
✅ SYB_14_9 — 247 rows
✅ SYB_15_1_1 — 208 rows
✅ SYB_15_1_2 — 243 rows
✅ SYB_15_1_3 — 2214 rows
✅ SYB_15_1_4 — 208 rows
✅ SYB_15_1_5 — 243 rows
✅ SYB_15_1_6 — 2214 rows
✅ SYB_16_2 — 156 rows
✅ SYB_16_3 — 864 rows
✅ SYB_17_10 — 117 rows
✅ SYB_17_11 — 972 rows
✅ SYB_17_1_1 — 13 rows
✅ SYB_17_1_2 — 13 rows
✅ SYB_17_4 — 130 rows
✅ SYB_17_6 — 322 rows
✅ SYB_17_7 — 364 rows
✅ SYB_18_0 — 13 rows
✅ SYB_18_1 — 26 rows
✅ SYB_18_2 — 130 rows
✅ SYB_18_3 — 1728 rows
✅ SYB_19_2_2 — 324 rows
✅ SYB_19_2_3 — 78 rows
✅ SYB_19_2_4 — 108 rows
✅ SYB_19_2_5 — 1350 rows
✅ SYB_20_4 — 66 rows
✅ SYB_20_5HIS — 198 rows

In [8]:
# Step 2 — Verify all tables and row counts
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print(f"Total tables in DB: {len(tables)}\n")
for tname in tables["name"]:
    count = pd.read_sql(f'SELECT COUNT(*) AS rows FROM "{tname}"', conn).iloc[0, 0]
    print(f"  {tname}: {count} rows")
conn.close()

Total tables in DB: 54

  SYB_11_1_10: 432 rows
  SYB_11_1_2: 324 rows
  SYB_11_1_3: 40 rows
  SYB_11_1_4: 1296 rows
  SYB_11_3_1: 54 rows
  SYB_11_3_2: 150 rows
  SYB_11_3_3: 90 rows
  SYB_13_1_1: 432 rows
  SYB_13_1_2: 540 rows
  SYB_13_1_3: 432 rows
  SYB_13_1_4: 288 rows
  SYB_13_1_5: 168 rows
  SYB_13_1_6: 2268 rows
  SYB_13_1_7: 1512 rows
  SYB_14_6: 144 rows
  SYB_14_7: 117 rows
  SYB_14_8: 44 rows
  SYB_14_9: 247 rows
  SYB_15_1_1: 208 rows
  SYB_15_1_2: 243 rows
  SYB_15_1_3: 2214 rows
  SYB_15_1_4: 208 rows
  SYB_15_1_5: 243 rows
  SYB_15_1_6: 2214 rows
  SYB_16_2: 156 rows
  SYB_16_3: 864 rows
  SYB_17_10: 117 rows
  SYB_17_11: 972 rows
  SYB_17_1_1: 13 rows
  SYB_17_1_2: 13 rows
  SYB_17_4: 130 rows
  SYB_17_6: 322 rows
  SYB_17_7: 364 rows
  SYB_18_0: 13 rows
  SYB_18_1: 26 rows
  SYB_18_2: 130 rows
  SYB_18_3: 1728 rows
  SYB_19_2_2: 324 rows
  SYB_19_2_3: 78 rows
  SYB_19_2_4: 108 rows
  SYB_19_2_5: 1350 rows
  SYB_20_4: 66 rows
  SYB_20_5HIS: 198 rows
  SYB_23_1: 180 ro

In [9]:
# Step 3 — Quick test query
conn = sqlite3.connect(DB_PATH)
df_test = pd.read_sql('SELECT * FROM "SYB_3_3" LIMIT 5', conn)
conn.close()
df_test

,اسم المؤشر,Measure Name,رمز المؤشر,Unit,الوحدة,Governorate,المحافظة,Sex,الجنس,قيمة المؤشر,سنة فترة القياس,شهر فترة القياس
0,لمواليد الأحياء المسجلون,Registered Live Births,M1,Number,عدد,Amman,العاصمة,Male,ذكر,38445,2023,None
1,لمواليد الأحياء المسجلون,Registered Live Births,M1,Number,عدد,Balqa,البلقاء,Male,ذكر,3842,2023,None
2,لمواليد الأحياء المسجلون,Registered Live Births,M1,Number,عدد,Zarqa,الزرقاء,Male,ذكر,9174,2023,None
3,لمواليد الأحياء المسجلون,Registered Live Births,M1,Number,عدد,Madaba,مأدبا,Male,ذكر,2095,2023,None
4,لمواليد الأحياء المسجلون,Registered Live Births,M1,Number,عدد,Irbid,إربد,Male,ذكر,17028,2023,None


In [10]:
# Step 4 — Build metadata table (table names, columns, sample values)
import openpyxl
import glob
import json

def read_cover_page(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["Cover Page"]
    info = {}
    all_items = []
    for row in ws.iter_rows(values_only=True):
        if row[0] and row[1]:
            info[str(row[0]).strip()] = str(row[1]).strip()
            all_items.append((str(row[0]).strip(), str(row[1]).strip()))
    wb.close()
    return info, all_items

def get_columns_and_samples(conn, sql_table):
    df_sample = pd.read_sql(f'SELECT * FROM "{sql_table}" LIMIT 3', conn)
    columns_info = ", ".join(df_sample.columns.tolist())
    sample_values = {}
    for col in df_sample.columns:
        uniq = pd.read_sql(
            f'SELECT DISTINCT "{col}" FROM "{sql_table}" WHERE "{col}" IS NOT NULL LIMIT 10',
            conn
        )
        sample_values[col] = uniq.iloc[:, 0].tolist()
    return columns_info, json.dumps(sample_values, ensure_ascii=False)

def build_metadata_table():
    excel_files = sorted(glob.glob("data/**/*.xlsx", recursive=True))
    rows = []
    conn = sqlite3.connect(DB_PATH)

    for path in excel_files:
        path = path.replace("\\", "/")
        if os.path.basename(path).startswith("~$"):
            continue
        sql_table = os.path.splitext(os.path.basename(path))[0]
        try:
            info, all_items = read_cover_page(path)
            name_ar = all_items[2][1] if len(all_items) > 2 else ""
            columns_info, sample_values = get_columns_and_samples(conn, sql_table)
            rows.append({
                "sql_table":     sql_table,
                "name_en":       info.get("Template Name", ""),
                "name_ar":       name_ar,
                "data_source":   info.get("Data Source", ""),
                "columns_info":  columns_info,
                "sample_values": sample_values,
            })
            print(f"✅ {sql_table}: {info.get('Template Name', '')}")
        except Exception as e:
            print(f"❌ {sql_table}: {e}")

    conn.close()
    df_meta = pd.DataFrame(rows)
    conn2 = sqlite3.connect(DB_PATH)
    df_meta.to_sql("metadata", conn2, if_exists="replace", index=False)
    conn2.close()
    print(f"\n✅ metadata table ready — {len(df_meta)} entries")
    return df_meta

df_meta = build_metadata_table()
df_meta[["sql_table", "name_en", "data_source", "columns_info"]].head(10)


✅ SYB_11_1_10: Number of Vehicles Involved in Road Accidents by Month and Type of Vehicle
✅ SYB_11_1_2: Lengths of Road Networks by Type of Road and Governorate
✅ SYB_11_1_3: Number of Licensed Vehicles and Percentage Change
✅ SYB_11_1_4: Number of Licensed  Vehicles by Type of Vehicle, Ownership and Governorate
✅ SYB_11_3_1: Local and External Mail
✅ SYB_11_3_2: Number of Post Boxes by Governorate
✅ SYB_11_3_3: Number of  Postal Service Centers by Governorate
✅ SYB_13_1_1: Number of Schools, Class Units, Students and Teachers  by Authority of Supervision and Sex schooling yea
✅ SYB_13_1_2: Number of Schools by Authority of Supervision, Educational Stage and Sex schooling year
✅ SYB_13_1_3: Number of Class Units by Authority of Supervision, Educational Stage and Sex schooling year
✅ SYB_13_1_4: Number of Students by Authority of Supervision, Educational Stage and Sex schooling year
✅ SYB_13_1_5: Number of Teachers by Authority of Supervision, Educational Stage and Sex schooling year
✅ 

,sql_table,name_en,data_source,columns_info
0,SYB_11_1_10,Number of Vehicles Involved in Road Accidents ...,Traffic Department,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
1,SYB_11_1_2,Lengths of Road Networks by Type of Road and G...,Ministry of Public Works & Housing,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
2,SYB_11_1_3,Number of Licensed Vehicles and Percentage Change,The Licensing Department,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
3,SYB_11_1_4,Number of Licensed Vehicles by Type of Vehicl...,The Licensing Department,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
4,SYB_11_3_1,Local and External Mail,Jordan Post Company,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
5,SYB_11_3_2,Number of Post Boxes by Governorate,Jordan Post Company,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
6,SYB_11_3_3,Number of Postal Service Centers by Governorate,Jordan Post Company,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
7,SYB_13_1_1,"Number of Schools, Class Units, Students and T...",Ministry of Education,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
8,SYB_13_1_2,"Number of Schools by Authority of Supervision,...",Ministry of Education,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."
9,SYB_13_1_3,Number of Class Units by Authority of Supervis...,Ministry of Education,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال..."


In [11]:
df_meta.head(10)

,sql_table,name_en,name_ar,data_source,columns_info,sample_values
0,SYB_11_1_10,Number of Vehicles Involved in Road Accidents ...,عدد المركبات المشتركة في حوادث الطرق حسب الشهر...,Traffic Department,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد المركبات المشتركة في حواد..."
1,SYB_11_1_2,Lengths of Road Networks by Type of Road and G...,أطوال شبكات الطرق الخارجية حسب نوع الطريق والم...,Ministry of Public Works & Housing,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""أطوال شبكات الطرق الخارجية""],..."
2,SYB_11_1_3,Number of Licensed Vehicles and Percentage Change,عدد المركبات المرخصة ونسبة التغير,The Licensing Department,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد المركبات المرخصة ونسبة ال..."
3,SYB_11_1_4,Number of Licensed Vehicles by Type of Vehicl...,عدد المركبات المرخصة حسب فئة المركبة وملكيتها...,The Licensing Department,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد المركبات المرخصة""], ""Mea..."
4,SYB_11_3_1,Local and External Mail,البريد الداخلي والخارجي,Jordan Post Company,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""البريد الداخلي والخارجي""], ""M..."
5,SYB_11_3_2,Number of Post Boxes by Governorate,عدد صناديق البريد حسب المحافظة,Jordan Post Company,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد صناديق البريد""], ""Measure..."
6,SYB_11_3_3,Number of Postal Service Centers by Governorate,عدد مراكز الخدمة البريدية حسب المحافظة,Jordan Post Company,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد مراكز الخدمة البريدية""], ..."
7,SYB_13_1_1,"Number of Schools, Class Units, Students and T...",عدد المدارس والشعب والطلبة والمعلمين حسب السلط...,Ministry of Education,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد المدارس والشعب والطلبة وا..."
8,SYB_13_1_2,"Number of Schools by Authority of Supervision,...",عدد المدارس حسب السلطة المشرفة والمرحلة التعلي...,Ministry of Education,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد المدارس""], ""Measure Name""..."
9,SYB_13_1_3,Number of Class Units by Authority of Supervis...,عدد الشعب حسب السلطة المشرفة والمرحلة التعليمي...,Ministry of Education,"اسم المؤشر, Measure Name, رمز المؤشر, Unit, ال...","{""اسم المؤشر"": [""عدد الشعب""], ""Measure Name"": ..."
